In [1]:
from pathlib import Path

import pandas as pd

BASE_PATH = Path("../../data/out/distillation/mmlu_distilled_deepseek_v4_flash_regenerate_incorrect_w_large.parquet")
REPLACEMENT_PATH = Path("../../data/out/distillation/mmlu_explained_answer_deepseek_v4_flash_extend_w_large.parquet")
OUT_PATH = Path("../../data/out/distillation/mmlu_distilled_w_explained_deepseek_v4_flash.parquet")

ID_COL = "question_id"
CORRECT_COL = "distill_ans_correct"

base_df = pd.read_parquet(BASE_PATH)
replacement_df = pd.read_parquet(REPLACEMENT_PATH)

print("Base shape:", base_df.shape)
print("Replacement shape:", replacement_df.shape)
print("Base correctness:\n", base_df[CORRECT_COL].value_counts(dropna=False))
print("Replacement correctness:\n", replacement_df[CORRECT_COL].value_counts(dropna=False))

Base shape: (12032, 14)
Replacement shape: (12032, 14)
Base correctness:
 distill_ans_correct
True     10637
False     1395
Name: count, dtype: int64
Replacement correctness:
 distill_ans_correct
True    12032
Name: count, dtype: int64


In [2]:
assert list(base_df.columns) == list(replacement_df.columns), "Column mismatch between base and replacement"
assert base_df[ID_COL].is_unique, "Base has duplicate question_id"
assert replacement_df[ID_COL].is_unique, "Replacement has duplicate question_id"

incorrect_ids = set(base_df.loc[base_df[CORRECT_COL] == False, ID_COL])
missing = incorrect_ids - set(replacement_df[ID_COL])
print(f"Incorrect rows in base: {len(incorrect_ids)}")
print(f"Incorrect ids missing from replacement: {len(missing)}")
assert not missing, f"Replacement is missing {len(missing)} ids needed for substitution"

Incorrect rows in base: 1395
Incorrect ids missing from replacement: 0


In [3]:
kept_df = base_df[~base_df[ID_COL].isin(incorrect_ids)]
replacement_rows = replacement_df[replacement_df[ID_COL].isin(incorrect_ids)]

merged_df = pd.concat([kept_df, replacement_rows], ignore_index=True)
merged_df = merged_df.set_index(ID_COL).loc[base_df[ID_COL]].reset_index()
merged_df = merged_df[base_df.columns.tolist()]

print("Merged shape:", merged_df.shape)
print("Merged correctness:\n", merged_df[CORRECT_COL].value_counts(dropna=False))
assert len(merged_df) == len(base_df)
assert (merged_df[ID_COL].values == base_df[ID_COL].values).all(), "Row order changed unexpectedly"

Merged shape: (12032, 14)
Merged correctness:
 distill_ans_correct
True    12032
Name: count, dtype: int64


In [4]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_parquet(OUT_PATH, index=False)
print(f"Wrote {len(merged_df)} rows to {OUT_PATH}")

Wrote 12032 rows to ../../data/out/distillation/mmlu_distilled_w_explained_deepseek_v4_flash.parquet
